In [1]:
%cd "E:/src code 2/python 2/KG"
import json
import pandas as pd
import os

from src.index.entity_extractor import EntityExtractor
from src.utils.config_loader import ConfigLoader
from src.llm.gemini import Gemini_LLM
from src.utils.utils import read_file
import os
from src.utils.type import TYPE_OF_ENTITY_IN_KG, TYPE_OF_JOB, TYPE_OF_JOB_ENTITY, TYPE_OF_CV, TYPE_OF_EDGE
from src.db.neo4j import GraphManager, Node, Edge
from src.query.utils import get_top_similar_node, get_father_node_name

gm = GraphManager("neo4j://localhost:7687", "neo4j", "123123aA@")
config = ConfigLoader().get_config_from_file(r"E:\src code 2\python 2\Legal_RAG\config\config.yaml")
llm = Gemini_LLM(config=config)

entities = ["Programming Language", "Library", "Software", "Technology", "Task", "Country", "City", "District"]


E:\src code 2\python 2\KG


In [23]:
cv = """TRAN CAO SON
 0363121538 ⋄ Dong Da, Hanoi
 caosonth2003@gmail.com
 OBJECTIVE
 AI engineer.
 EDUCATION
 Bachelor of Computer Science, Hanoi University of Science and Technology (HUST)
 CPA: 3.7
 SKILLS AND TECHNICAL STRENGTHS
 2021- Expected 2025
 Programming Languaes
 Database
 Computer science
 AI
 Other
 EXPERIENCE
 Python, C, C++, Java.
 MySQL, MongoDB, Neo4j, ChromaDB, Redis.
 Data Structures and Algorithms, Computer Architecture, OS, Networks.
 Strong knowledge in ML, DL, LLM, VLM, CV and NLP.
 Experienced in using LLAMA, WEAVIATE, MIVUS, CHROMA
 Experienced in using frameworks such as PyTorch and TensorFlow.
 Experienced in RAG, GraphRAG, deploy and finetune LLM, text embedding.
 Image/audio processing, deploy (docker, triton, vLLM, fastAPI), MinIO.
 Data scientist
 Viettel Telecom
 • Developing a recommendation system for the TV360 application (currently).
 June 2024- Present
 Hanoi
 • Developed the Mydio chatbot for the Mydio app using LLM, and deployed it with vLLM and Triton.
 • Developed pdf to audio api, training TTS model, OCR model and using LLM to improve multilingual speech.
 AI Engineer
 Kaopiz
 • Training OCR model, deploy, quantize, image processing.
 • Improve accuracy and speed of model.
 Jan 2024- June 2024
 Hanoi
 PROJECTS
 Mydio chatbot Built a chatbot can open books, search for books, control playback, process payments, open
 favorites, and much more. Increase speed and accuracy by utilizing multi-agent systems with a hierarchical
 architecture, leveraging vLLM and Triton to accelerate inference. Using Qwen 2.5 14B and Vietnamese text
 encoder.
 Mydio audiobooks Build a high-quality speech synthesis using text to speech. This project using Fastpitch model
 (TTS) and Qwen 2.5 14B (LLM) to convert word into ipa. This model can speech in multilingual in high quality.
 OCR for Japanese Build a OCR model with 98-99% accuracy on hand written character.
 EXTRA-CURRICULAR ACTIVITIES
 • Teaching assistant for the course ”Applied Algorithms”.
 AWARDS
 • Top 1 in Vietnam for A00 group (Mathematics, Physics, and Chemistry).
 • The scholarship for academic encouragement at Hanoi University of Science and Technology.
 • Top 6 of SOICT Hackathon 2024 track legal document retrieval"""

In [24]:
entity_extractor = EntityExtractor(entities=", ".join(entities), llm=llm)
entities_list, cv_exp  = entity_extractor.get_entities_from_cv(cv=cv, entity_types=", ".join(entities))
list_node = [e.node() for e in entities_list]
for node in list_node:
    node.label = TYPE_OF_JOB_ENTITY
cv_node = Node(TYPE_OF_CV, {"name":"Trần Cao Sơn CV"})
top_jobs, score = get_top_similar_node(gm, cv_node, list_node, TYPE_OF_JOB)

In [25]:
top_jobs

['backend_2_v3', 'ai_ai_3_v3', 'ai_ai_10_v3', 'ai_ai_4_v3', 'ai_ai_5_v3']

In [26]:
for e in entities_list:
    print(e.entity_name)

PYTHON
C
C++
JAVA
MYSQL
MONGODB
NEO4J
CHROMADB
REDIS
PYTORCH
TENSORFLOW
LLAMA
WEAVIATE
MIVUS
CHROMA
DOCKER
TRITON
VLLM
FASTACKAPI
MINIO
Data Structures and Algorithms
Computer Architecture
OS
Networks
ML
DL
LLM
VLM
CV
NLP
RAG
GraphRAG
text embedding
Image/audio processing


In [34]:
def get_exp(top_jobs):
    job2exp = {}
    for job in top_jobs:
        folder = "E:/data/job2/" + job.split("_")[0]
        file = "_".join(job.split("_")[1:]) + ".txt"
        with open(f"{folder}/{file}", 'r', encoding="utf-8") as f:
            data = f.read().split("\n")
            exp_line = [line for line in data if line.lower().startswith("exp")][0]
            job_exp = float(exp_line.split(":")[1])
            # print(job, job_exp)
            job2exp[job] = job_exp
    return job2exp

In [35]:
job2exp = get_exp(top_jobs)
for job in top_jobs:
    if job2exp[job] <= cv_exp:
        print(job)

backend_2_v3
ai_ai_3_v3
ai_ai_10_v3
ai_ai_4_v3
ai_ai_5_v3


In [36]:
job2exp[job]

0.0

In [37]:
top_jobs

['backend_2_v3', 'ai_ai_3_v3', 'ai_ai_10_v3', 'ai_ai_4_v3', 'ai_ai_5_v3']